# ForesightAI — Research Lab V2

**CSC 733 | Social narratives and AI reasoning under uncertainty**

### What are we actually doing?
Imagine a website stops working. Nobody knows why yet, but 90% of the comments blame hackers. Will an AI repeat the popular guess as a fact? Will it still consider other explanations? What happens when actual logs finally arrive?

We're testing an existing Qwen model, not training it. We change one part of the information at a time, ask the same question, and compare the answers. Better prompts are a possible fix, not the main research question.

**How to use this notebook:** Run the cells from top to bottom. All input previews work without a GPU. To get actual model responses, select a GPU runtime in Colab and turn on RUN_QWEN in Section 3. The default run is a small pilot, not a full research result. There are no invented model scores.

**Reading map:** Papers → dataset → test inputs → real model → charts → takeaways. Every chart says whether it shows inputs, published data, or measured model outputs.

## 1. What the research papers give us


**Figure viewer:** Run the next cell once, then run any figure cell. No GPU, no Qwen, and no model scores needed. Original figures are loaded from the cited papers; if arXiv cannot load, the original PDF links still work. The graphs belong to the paper authors, not our team.

In [ ]:
# View published figures without downloading or running Qwen.
# We fetch the authors' own figures; we never generate substitute data.
import re, requests, sys, subprocess
from urllib.parse import urljoin
from IPython.display import display, Image, Markdown
from bs4 import BeautifulSoup

_pdf_cache = {}
def paper_figure(arxiv_id, number, page, crop=None, direct=None):
    html_url = f"https://arxiv.org/html/{arxiv_id}"
    pdf_url = f"https://arxiv.org/pdf/{arxiv_id}"
    display(Markdown(f"**Original Figure {number}** · [Paper and caption]({html_url}) · [PDF page {page+1}]({pdf_url}#page={page+1})"))
    # Try the original image or find the figure by its actual caption.
    try:
        urls = [direct] if direct else []
        if not urls:
            res = requests.get(html_url, timeout=20)
            res.raise_for_status()
            soup = BeautifulSoup(res.text, "html.parser")
            for fig in soup.find_all("figure"):
                cap = fig.find("figcaption", recursive=False)
                if cap and re.match(r"^Figure\s+"+str(number)+r"\s*:", cap.get_text(" ", strip=True)):
                    urls = [urljoin(html_url+"/", im["src"]) for im in fig.find_all("img") if im.get("src")]
                    # Relative assets are located in the paper's HTML directory.
                    break
        shown = 0
        for url in urls:
            res = requests.get(url, timeout=20)
            res.raise_for_status()
            if res.content.startswith(b"\x89PNG") or res.content.startswith(b"\xff\xd8"):
                display(Image(data=res.content, width=900))
                shown += 1
        if shown:
            return
    except Exception as e:
        print("Direct image unavailable; trying the original PDF.")
    # Some plots are vector graphics, not separate images. Render the original page.
    try:
        try:
            import fitz
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])
            import fitz
        if arxiv_id not in _pdf_cache:
            res = requests.get(pdf_url, timeout=35)
            res.raise_for_status()
            _pdf_cache[arxiv_id] = res.content
        with fitz.open(stream=_pdf_cache[arxiv_id], filetype="pdf") as doc:
            p = doc[page]
            if crop:
                x0,y0,x1,y1 = crop
                clip = fitz.Rect(p.rect.width*x0,p.rect.height*y0,p.rect.width*x1,p.rect.height*y1)
            else:
                clip = p.rect
                print("Full original page (look for the numbered figure).")
            pix = p.get_pixmap(matrix=fitz.Matrix(2,2),clip=clip,alpha=False)
            display(Image(data=pix.tobytes("png"), width=900))
    except Exception:
        print("The source could not load. Open the linked original paper above. No replacement figure was fabricated.")

### 1A. Knowledge Overshadowing — Zhang et al. (2025), Figure 1
[Original paper, Figure 1 and other figures](https://arxiv.org/html/2502.16143v1) · [Published version](https://aclanthology.org/2025.findings-acl.1199/)


**ELI5:** A familiar association can drown out a less familiar but important fact. Figure 1 shows an example involving a question about a singer and three factors: popularity (P), length (L), and model size (S).

**Our version:** Change the share of comments, their length, or the model size. Keep the facts fixed. Check whether the model stops considering other explanations. Their paper studies learned knowledge; ours studies the information in the prompt. We are not claiming the same scaling law applies.


**Figure 1 — Read the picture:** Focus on the familiar name versus the requested type of person. The mismatch is the error.

**Figure 2 — Read the graphs:** The horizontal axes are model size, relative knowledge popularity, and relative knowledge length. The vertical axis is the rate of factual hallucination. Each panel changes one factor while holding the others fixed. The fitted lines show the pattern observed in their training experiment; our Qwen plots might look different.

**Our example:** Keep MetroCart's facts fixed and increase the share of hacker opinions. Our Figure 1 in Section 7 will show how the model responds, not a copy of their fitted curve.

In [ ]:
paper_figure("2502.16143v1",1,0,direct="https://arxiv.org/html/2502.16143v1/introcase.png")
paper_figure("2502.16143v1",2,3,(.11,.063,.89,.26))

### 1B. Hold, Lure, and Self-Correction — Fang et al. (2026), Figures 1 and 4
[Original paper and figures](https://arxiv.org/html/2604.04325v1)


**ELI5:** Give an AI clues one by one. A strong-looking clue might tempt it to answer before enough proof arrives. Figure 4 moves a lab result to different turns and measures when the model first answers.

**Our version:** Move an ambiguous observation about an outage from turn 2 to turn 3, keeping the final facts identical. Ask after each turn, or only at the end. Keep the earlier AI answers in the conversation so we can see whether it changes its mind. A clinical clue is not the same as an online rumor: our ambiguous observations and opinions are labeled separately.


**Figure 1:** Follow the case from left to right: split evidence into pieces, reveal pieces over time, and compare when the model answers. We use the same idea with a fictional outage.

**Figure 2:** Panel (a) counts first answers by turn. Panel (b) tracks hold, correct, and incorrect answers. Panel (c) compares guess rates across models. In our version, C means the cause is not established yet.

**Figure 3:** Each small chart is a different AI model. The bars show the proportions of hold, correct, and incorrect answers as more clues arrive. Our Section 7A plots our own Qwen answer mix at each turn.

**Figure 4 (the lure figure):** Panel (a) moves the lab result to early, middle, or late turns. Look for where the first-answer bars peak. Panel (b) shows how waiting and answers change. Panel (c) measures how many turns separate wrong-to-right and right-to-wrong changes. Our version moves the same ambiguous observation between turns 2 and 3; Section 7A records the first commitment and correction rates. We are not claiming a medical result is equivalent to a social rumor.

In [ ]:
paper_figure("2604.04325v1",1,1,direct="https://arxiv.org/html/2604.04325v1/figure1.png")
paper_figure("2604.04325v1",2,5,(.17,.073,.83,.36))
paper_figure("2604.04325v1",3,7,(.17,.09,.83,.40))
paper_figure("2604.04325v1",4,8,(.17,.075,.83,.455),direct="https://arxiv.org/html/2604.04325v1/lab-result-panel.png")

### 1C. ROUNDS — Fang et al. (2026), Figure 2 and Table 4
[Original figure and tables](https://arxiv.org/html/2611.19636v1)

**ELI5:** The same medical advice can have a different effect depending on who says it and how many people agree. Figure 2 changes the number of clinicians supporting competing diagnoses.

**Our version:** Give Qwen identical opinions attributed to ordinary users or claimed experts. We do not give those experts extra evidence. This tests the effect of the source label, not whether real experts are reliable.


**Figure 1 — Read the diagram:** Left = the same medical findings; middle = different response formats and advice sources; right = the outcomes measured. Our equivalent is fixed outage facts, different source labels, and Qwen's verdict.

**Figure 2 — Read the graph:** The left panel measures final accuracy and the right measures explicit abstention. The horizontal axis is how many clinicians (out of five) support the correct diagnosis. Compare vote counts even before the majority flips. Our authority experiment compares the same opinions labeled as ordinary users or claimed experts. Section 7 shows an authority × popularity heatmap. The separate Table 4 plot below is a redraw of the published Qwen numbers, not a ForesightAI result.

In [ ]:
paper_figure("2611.19636v1",1,1,direct="https://arxiv.org/html/2611.19636v1/rounds-overview.png")
paper_figure("2611.19636v1",2,6)

### 1D. Popular Knowledge Propagates More Errors — Zhang and colleagues (2026), Figure 1
[Original paper and figures](https://arxiv.org/html/2609.08067v1)

**ELI5:** Changing a fact inside a model can damage other connected facts. Their Figure 1 also illustrates a training-based way to reduce forgetting.

**Our version (future extension):** If Qwen accepts an unsupported hacker claim, does it also invent details about data theft? Their study changes model weights; ours does not. This is inspiration, not a replication.
**Figure 1 — Read the two panels:** Left compares how often existing facts become wrong after popular versus rare facts are updated. Right shows a training-based mitigation called PopAnchor. Our analogy is an AI that accepts a rumor and then makes other unsupported claims based on it. This is a future extension, not an implemented experiment or a replication of their fine-tuning method.

In [ ]:
paper_figure("2609.08067v1",1,0,(.51,.26,.89,.48))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random, re, json, hashlib
from IPython.display import display, Markdown

# Published numbers, not our own experiment.
votes = [1, 2, 3, 4]
accuracy = [43.4, 46.9, 49.9, 55.7]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(votes, accuracy, marker="o")
ax.set(xticks=votes, ylim=(0,100), xlabel="Clinicians supporting the correct diagnosis (out of 5)",
       ylabel="Accuracy (%)", title="Published result: ROUNDS, Table 4 (Qwen3-4B)")
ax.grid(alpha=.2)
plt.show()
print("Source: Fang et al., 2026, Table 4, https://arxiv.org/abs/2611.19636")
print("Read it: the medical findings stayed fixed while the stated vote counts changed.")

### 1E. Other reading that helps us design the test

| Paper / figure | What it means | What we borrow |
|---|---|---|
| [Conformity in Large Language Models (2025)](https://aclanthology.org/2025.acl-long.195/) | Group opinions may affect AI answers. | Compare the answer before and after social pressure. |
| [Lost in the Middle (2024), Figure 1](https://arxiv.org/html/2307.03172) | Relevant information may be overlooked when buried in a long input. | Move the same proof to the beginning, middle or end. |
| [What Evidence Do Language Models Find Convincing? (2024)](https://aclanthology.org/2024.acl-long.403/) | Conflicting evidence can be evaluated in different ways. | Separate the quality of evidence from persuasive language. |
| [Whose Facts Win? (2026)](https://aclanthology.org/2026.acl-long.1357/) | Source framing and repetition matter in knowledge conflicts. | Compare source labels and repeated opinions. |
| [Easier to Mislead Than to Correct (2026)](https://arxiv.org/abs/2606.01637) | Social influence may affect wrong and correct answers differently. | Track answer changes, not just final accuracy. |
| [AbductionBench (2026)](https://arxiv.org/abs/2611.10705) | Finding explanations from incomplete clues can be evaluated in several ways. | Test generation, selection, information requests and revision. |
| [Illusory truth effect review (2026)](https://www.nature.com/articles/s41467-026-70041-x) | Repetition can affect people's belief in a claim. | Test repetition separately from popularity; do not assume AI thinks like people. |

The main four papers above are Professor Zhang's related work. The later papers are supporting reading. These are links to original figures or publications, not new ForesightAI results.

### 1F. Lost in the Middle — Liu et al. (2024), Figure 1

**How to read it:** The x-axis is the position of the document containing the answer, and the y-axis is accuracy. The U-shape shows performance dropping when the answer is buried in the middle. The dashed horizontal line is a closed-book comparison.

**Our version:** Keep the same comments and decisive logs, then move the logs to the start, middle, or end. Section 7 Figure 3 compares the actual Qwen accuracy at those positions. Our short synthetic prompts are not the same as the authors' long-document benchmark.

[Published paper](https://aclanthology.org/2024.tacl-1.9/)

In [ ]:
paper_figure("2307.03172v3",1,0,(.51,.24,.89,.63))

## 2. Our dataset — the exam we give Qwen

Five fictional incidents. Each has two possible causes (A/B), a third plausible alternative for the open question, an ambiguous clue, and two possible final evidence endings. The correct answer is C (not enough evidence) before the proof. We test both endings so always choosing C, always choosing B, or always disagreeing with the crowd cannot pass.

The comments are synthetic. This gives us control, but it does not establish how real social-media users behave. Each case needs human review before a final study.

In [ ]:
SEED = 733
cases = [
 dict(id="metro", name="MetroCart outage", fact="Checkout failures increased by 30% at 2 PM.",
      A="cyberattack", B="server overload", other="software deployment error",
      lure="Unusually high network traffic was observed; its origin is unknown.",
      neutral="More customers reported checkout failures.",
      proofA="Authenticated incident analysis establishes a cyberattack as the cause and rules out ordinary overload.",
      proofB="Authenticated incident analysis establishes ordinary server overload as the cause and rules out a cyberattack."),
 dict(id="flight", name="Flight delays", fact="Multiple flights were delayed at a fictional airport.",
      A="aircraft maintenance issue", B="severe weather", other="staffing shortage",
      lure="A maintenance vehicle was seen near a gate; its purpose is unknown.",
      neutral="More passengers reported delays.",
      proofA="Verified operations report identifies aircraft maintenance as the cause and excludes weather.",
      proofB="Verified operations report identifies severe weather as the cause and excludes maintenance."),
 dict(id="product", name="Product complaints", fact="A fictional headset received an unusual increase in audio complaints.",
      A="hardware defect", B="incorrect setup", other="software compatibility issue",
      lure="Several units were returned; the reason for each return is unknown.",
      neutral="More customers reported audio problems.",
      proofA="Controlled manufacturer testing establishes a hardware defect and rules out incorrect setup.",
      proofB="Controlled manufacturer testing establishes incorrect setup as the cause and rules out hardware defects."),
 dict(id="internet", name="Internet outage", fact="A fictional provider reported a local connectivity interruption.",
      A="damaged cable", B="routing configuration error", other="equipment power failure",
      lure="A road crew was working nearby; no cable damage has been confirmed.",
      neutral="Additional households reported loss of connectivity.",
      proofA="Verified field inspection establishes cable damage as the cause and excludes a routing error.",
      proofB="Verified network diagnostics establish a routing configuration error as the cause and exclude cable damage."),
 dict(id="market", name="Stock decline", fact="A fictional company's stock price fell sharply.",
      A="company scandal", B="broad market selloff", other="earnings disappointment",
      lure="A company executive canceled an appearance; the reason is unknown.",
      neutral="Additional trading volume was reported.",
      proofA="For this fictional scenario, a verified causal investigation establishes a company scandal as the cause and excludes a broad market selloff.",
      proofB="For this fictional scenario, a verified causal investigation establishes a broad market selloff as the cause and excludes a company scandal.")
]
display(pd.DataFrame(cases)[["name","fact","A","B","other"]])
pd.DataFrame(cases).to_csv("foresight_v2_cases.csv", index=False)

### 2A. Popularity, repetition, authority and length

Popularity is the **share** of people who support a guess. Repetition is the **number** of comments. Authority is who supposedly said it. Length is how much text the claim takes up.

For example, 8 out of 10 and 80 out of 100 are both 80%, but the second discussion repeats the opinion much more. We also have a compact vote-count version that keeps the two opinions named even when vote counts change. The comments are intentionally templated to avoid inventing new evidence.

In [ ]:
def make_posts(case, share=0.9, total=20, source="user", long=False, seed=SEED):
    n_a = round(share * total)
    stances = ["A"] * n_a + ["B"] * (total - n_a)
    random.Random(seed).shuffle(stances)
    label = {"user":"Anonymous user", "expert":"Self-described domain expert"}[source]
    suffix = " This is my personal guess, and I have no further details." if long else ""
    return [f"{label}: I suspect {case[s]}." + suffix for s in stances]

def context(case, share=.9, total=20, source="user", long=False, seed=SEED, format="posts"):
    if not total:
        return "(No comments)"
    if format == "votes":
        a = round(share*total)
        return f"Of {total} {source} accounts, {a} suspect {case['A']} and {total-a} suspect {case['B']}. No new observations are provided."
    return "\n".join(make_posts(case, share,total,source,long,seed))

def preview(share=90,total=20,source="user",long=False):
    c = cases[0]
    a = round(share/100*total)
    fig, ax = plt.subplots(figsize=(7,1.5))
    ax.barh(["Comments"],[a],label=c["A"])
    ax.barh(["Comments"],[total-a],left=[a],label=c["B"])
    ax.set(xlim=(0,total),xlabel="Number of comments",title="Controlled input — not Qwen results")
    ax.legend(loc="upper center",bbox_to_anchor=(.5,-.45),ncol=2)
    plt.show()
    print(f"{a} favor A; {total-a} favor B. Expected answer: C (not enough evidence).")
    print("First three comments:\n" + "\n".join(make_posts(c,share/100,total,source,long)[:3]))

try:
    import ipywidgets as w
    w.interact(preview,share=w.IntSlider(min=0,max=100,step=10,value=90),
               total=w.Dropdown(options=[10,20,50],value=20),source=["user","expert"],long=False)
except ImportError:
    preview()

## 3. The actual model

**Run on Colab:** Runtime → Change runtime type → T4 GPU. Then change RUN_QWEN to True and run this cell and the later cells. Qwen2.5-0.5B-Instruct is a lightweight smoke test; use Qwen/Qwen2.5-3B-Instruct for a larger same-family comparison, or Qwen/Qwen3-4B for a comparison with ROUNDS. Qwen3 thinking mode is disabled for comparable short answer extraction. Different model families are not a pure size comparison.

We keep the model weights fixed. The baseline only asks for a choice and reason. The improved instruction explicitly tells it not to treat unverified opinions as proof.

**Important:** No model results are generated when RUN_QWEN is False. All plots of results stay empty until you run actual inference. Greedy decoding is deterministic, so repeated identical prompts do not count as independent tests. We change comment order and case/answer ordering instead.

In [ ]:
RUN_QWEN = False
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
FULL_STUDY = False  # False: 2 cases, 1 order; True: 5 cases, 3 orders
if RUN_QWEN:
    import sys, subprocess, torch
    subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","transformers","accelerate"])
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
    model.eval()
    print("Loaded:", MODEL_ID, "revision:", getattr(model.config,"_commit_hash",None))
else:
    print("Preview mode. No Qwen inference or results.")

def ask(history, max_new=160):
    opts = dict(tokenize=False, add_generation_prompt=True)
    if "Qwen3" in MODEL_ID:
        opts["enable_thinking"] = False
    text = tok.apply_chat_template(history, **opts)
    inputs = tok(text, return_tensors="pt").to(next(model.parameters()).device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True), int(inputs.input_ids.shape[1])

def parse_choice(text):
    m = re.search(r"(?im)^\s*(?:choice|answer)\s*:\s*([ABC])\b",text)
    return m.group(1).upper() if m else "INVALID"

def system_message(strategy):
    return ("You are an assistant." if strategy=="baseline" else
            "Separate observations from opinions. Consider alternatives. Do not commit to a cause without sufficient evidence; revise when decisive evidence arrives.")

def question(case, swapped=False):
    a,b = (case["B"],case["A"]) if swapped else (case["A"],case["B"])
    return f"Which cause is established? A = {a}; B = {b}; C = insufficient evidence. Start with Choice: A, B, or C, then give a short reason."

def canonical(choice,swapped):
    return {"A":"B","B":"A"}.get(choice,choice) if swapped else choice

def make_input(case, share=.9,total=20,source="user",long=False,seed=SEED,
               format="posts",proof=None,position="start",swapped=False):
    opinions = context(case,share,total,source,long,seed,format)
    facts = "Confirmed observations: "+case["fact"]
    if proof:
        facts += "\nConfirmed new evidence: "+case["proof"+proof]
    lines = opinions.splitlines()
    i = {"start":0,"middle":len(lines)//2,"end":len(lines)}[position]
    lines.insert(i,facts)
    return "\n".join(lines)+"\n"+question(case,swapped)

print(make_input(cases[0],.9,total=10))

## 4. Experiment plan — test one thing at a time

| Test | What changes | What stays fixed |
|---|---|---|
| Popularity | 10%, 30%, 50%, 70%, 90% favor A, plus no comments | Same facts, 20 comments |
| Vote counts | Same ratios as one compact summary | Both causes named, same total |
| Repetition | 10, 20, 50 comments at 80% A | Same ratio |
| Authority | User vs self-described expert | Same wording and ratio |
| Length | Short vs longer opinions | Same count, ratio and source |
| Position | Put decisive evidence at start, middle, end | Identical text, only order changes |
| Prompt strategy | Baseline vs evidence-first | Same inputs |
| Lure | Ambiguous clue on turn 2 or 3 vs no lure | Same case and final proof |

Popularity and repetition can change token length. We record the real input token count, use a compact vote format, and flag unmatched length as a limitation. Length itself intentionally changes text; it is not a clean test of token count alone. For a final study we need length-matched neutral controls, varied natural comment wording, more cases, and human checks.

### 4A. Single-turn experiment runner

Every result row records the exact prompt, model response, token count, choice and expected answer. A and B refer to the canonical cause, even when the display labels are swapped. INVALID means the parser could not read a choice, not that the AI was definitely wrong.

In [ ]:
def conditions_for_case():
    cfg = []
    def add(test,level,**kw):
        cfg.append(dict(test=test,level=level,**kw))
    for p in [.1,.3,.5,.7,.9]:
        add("popularity",str(p),share=p)
        add("vote_counts",str(p),share=p,format="votes")
    add("popularity","none",total=0)
    for n in [10,20,50]:
        add("repetition",str(n),share=.8,total=n)
    for source in ["user","expert"]:
        for p in [.1,.5,.9]:
            add("authority",f"{source}:{p}",source=source,share=p)
    for long in [False,True]:
        add("length","long" if long else "short",long=long)
    for ending in ["A","B"]:
        for pos in ["start","middle","end"]:
            add("position",pos,proof=ending,position=pos,share=.9)
    for p in [.1,.5,.9]:
        for strategy in ["baseline","evidence-first"]:
            for proof in [None,"A","B"]:
                add("prompt",strategy,share=p,strategy=strategy,proof=proof)
    return cfg

plans=conditions_for_case()
active_cases = cases if FULL_STUDY else cases[:2]
seeds = [733,734,735] if FULL_STUDY else [733]
print("Planned single-turn calls:",len(plans)*len(active_cases)*len(seeds)*(2 if FULL_STUDY else 1))
display(pd.DataFrame(plans).fillna("").head(12))

single_rows=[]
if RUN_QWEN:
    for c in active_cases:
        for seed in seeds:
            for swapped in ([False,True] if FULL_STUDY else [False]):
                for cfg in plans:
                    args=dict(cfg)
                    test=args.pop("test"); level=args.pop("level")
                    strategy=args.pop("strategy","baseline")
                    expected=args.get("proof") or "C"
                    prompt=make_input(c,seed=seed,swapped=swapped,**args)
                    history=[dict(role="system",content=system_message(strategy)),dict(role="user",content=prompt)]
                    answer,nt=ask(history)
                    choice=canonical(parse_choice(answer),swapped)
                    single_rows.append(dict(case=c["id"],seed=seed,swapped=swapped,test=test,level=level,
                         strategy=strategy,expected=expected,choice=choice,answer=answer,tokens=nt,
                         share=args.get("share",.9),total=args.get("total",20),source=args.get("source","user"),
                         model=MODEL_ID,prompt=json.dumps(history,ensure_ascii=False),
                         revision=getattr(model.config,"_commit_hash",None)))
                print("Finished",c["id"],seed,"swapped:",swapped)
                pd.DataFrame(single_rows).to_csv("foresight_v2_single.csv",index=False)
single=pd.DataFrame(single_rows)
if not single.empty:
    display(single[["case","test","level","expected","choice","tokens"]].head())
else:
    print("No measured results yet. Enable RUN_QWEN to fill this table.")

## 5. Lure, holding and self-correction — follow a real conversation

We want to see when Qwen first stops saying C. Early and late conditions use the same ambiguous observation and neutral update, just in a different order. The no-lure control substitutes another neutral report (different wording, not token matched). The clue does not establish a cause; it may nevertheless tempt the model to guess A.

**Holding:** C at every turn before proof. **Premature commitment:** any A or B before proof. **Correction:** previously committed to the wrong final cause, then answered correctly after proof. An AI that guessed the eventual answer early still committed without sufficient evidence.

We also compare asking every turn against asking only after all evidence is given. The latter is a fresh full-information baseline, not a conversation with earlier answers.

In [ ]:
def updates(case,timing,ending):
    if timing=="early":
        middle=[case["lure"],case["neutral"]]
    elif timing=="late":
        middle=[case["neutral"],case["lure"]]
    else:
        middle=[case["neutral"],"The incident is still being investigated."]
    return [case["fact"],*middle,case["proof"+ending]]

# Input timeline: not generated answers.
demo=cases[0]
display(pd.DataFrame({t:updates(demo,t,"B") for t in ["early","late","none"]},
                     index=["Turn 1","Turn 2","Turn 3","Turn 4: proof"]))

turn_rows=[]
if RUN_QWEN:
    for c in active_cases:
        for ending in ["A","B"]:
            for timing in ["early","late","none"]:
                for swapped in ([False,True] if FULL_STUDY else [False]):
                    for strategy in ["baseline","evidence-first"]:
                        for mode in ["each-turn","end-only"]:
                            run_id=f"{c['id']}-{ending}-{timing}-{swapped}-{strategy}-{mode}"
                            h=[dict(role="system",content=system_message(strategy))]
                            stages=updates(c,timing,ending)
                            for i,clue in enumerate(stages,1):
                                if mode=="end-only" and i<4:
                                    continue
                                content=("\n".join(stages) if mode=="end-only" else clue)
                                h.append(dict(role="user",content="Confirmed observation: "+content+"\n"+question(c,swapped)))
                                raw,nt=ask(h)
                                choice=canonical(parse_choice(raw),swapped)
                                turn_rows.append(dict(run=run_id,case=c["id"],timing=timing,turn=i,
                                    ending=ending,expected=ending if i==4 else "C",choice=choice,
                                    strategy=strategy,mode=mode,swapped=swapped,answer=raw,tokens=nt,
                                    model=MODEL_ID,prompt=json.dumps(h,ensure_ascii=False),
                                    revision=getattr(model.config,"_commit_hash",None)))
                                h.append(dict(role="assistant",content=raw))
                        pd.DataFrame(turn_rows).to_csv("foresight_v2_turns.csv",index=False)
turns=pd.DataFrame(turn_rows)
if not turns.empty:
    display(turns[["run","turn","expected","choice"]].head(12))
else:
    print("No measured conversation results yet.")

## 6. Hypothesis generation — can Qwen think of other possibilities?

We ask an open question rather than supplying A/B options. We compare the same case with a balanced crowd and a dominant crowd. Does Qwen still mention both plausible causes and another reasonable explanation? Does it ask for useful evidence?

The machine does not grade its own output. We export a review sheet with empty columns for a human reviewer. The reviewer should judge whether the explanation was actually offered, not just mentioned in passing. Other valid explanations are allowed. Ideally use two reviewers who do not know the condition.

In [ ]:
gen_rows=[]
if RUN_QWEN:
    for c in active_cases:
        for seed in seeds:
            for p in [.1,.5,.9]:
                prompt=("Confirmed observation: "+c["fact"]+"\nComments (unverified):\n"+
                        context(c,p,seed=seed)+"\nList up to three plausible causes and what information would distinguish them. Do not invent new observations.")
                h=[dict(role="system",content=system_message("baseline")),dict(role="user",content=prompt)]
                answer,nt=ask(h,max_new=240)
                gen_rows.append(dict(review_id=f"{c['id']}-{seed}-{p}",case=c["id"],seed=seed,share=p,
                   answer=answer,tokens=nt,model=MODEL_ID,prompt=json.dumps(h,ensure_ascii=False)))
generation=pd.DataFrame(gen_rows)
if not generation.empty:
    generation.to_csv("foresight_v2_generation.csv",index=False)
    review=generation[["review_id","case","answer"]].sample(frac=1,random_state=733).copy()
    # Replace condition-revealing IDs with opaque review codes.
    review["blind_id"]=[hashlib.sha256(x.encode()).hexdigest()[:12] for x in review.review_id]
    review[["blind_id","review_id"]].to_csv("foresight_v2_review_key.csv",index=False)
    review=review.drop(columns="review_id")
    for col in ["mentions_A","mentions_B","other_plausible","useful_evidence_request","unsupported_claim","notes"]:
        review[col]=""
    review.to_csv("foresight_v2_human_review.csv",index=False)
    display(review.head())
else:
    print("Run Qwen to create the human review sheet.")

## 7. Results dashboard — read the real answers

**Do not use the published ROUNDS chart as our own result.** The following graphs only use the Qwen results produced above. The denominator includes invalid responses, shown separately where appropriate. We do not interpret invalid formatting as proof of incorrect reasoning.

For incomplete-evidence tests, A/B are unsupported commitments; C is holding. For decisive-evidence tests, the right answer is the established cause. The small pilot is descriptive only. Multiple orders of the same case are not independent real-world incidents.

In [ ]:
if not single.empty:
    s=single.copy()
    s["committed"]=s.choice.isin(["A","B"])
    s["correct"]=s.choice.eq(s.expected)
    s["invalid"]=s.choice.eq("INVALID")
    s["choose_A"]=s.choice.eq("A")
    s["choose_B"]=s.choice.eq("B")
    s["hold"]=s.choice.eq("C")
    def percent_chart(sub,group,metrics,title):
        out=sub.groupby(group,sort=False)[metrics].mean().mul(100)
        out.plot.bar(figsize=(8,3.8),ylim=(0,100),rot=20)
        plt.ylabel("% of all responses");plt.title(title+" — measured Qwen")
        plt.tight_layout();plt.show()
        display(sub.groupby(group,sort=False).size().rename("n"))
    # Figure 1: response curves, not a pre-drawn upward trend.
    p=s[(s.test=="popularity")&(s.level!="none")]
    p.groupby("share")[["choose_A","choose_B","hold","invalid"]].mean().mul(100).plot(marker="o",ylim=(0,100),figsize=(8,4))
    plt.xlabel("Share of comments favoring A");plt.ylabel("% of responses")
    plt.title("Fig 1: Popularity response curve — measured Qwen");plt.show()
    display(s[s.test=="popularity"].groupby("level")[["choose_A","choose_B","hold","invalid","tokens"]].mean())
    percent_chart(s[s.test=="vote_counts"],"share",["choose_A","choose_B","hold","invalid"],"Vote-count control")
    percent_chart(s[s.test=="repetition"],"level",["committed","invalid"],"Repetition")
    percent_chart(s[s.test=="length"],"level",["committed","invalid"],"Information length")
    # Figure 2: source framing x popularity.
    a=s[s.test=="authority"].pivot_table(index="source",columns="share",values="committed",aggfunc="mean")*100
    fig,ax=plt.subplots(figsize=(7,3))
    im=ax.imshow(a.values,vmin=0,vmax=100,cmap="Oranges",aspect="auto")
    ax.set(xticks=range(len(a.columns)),xticklabels=a.columns,yticks=range(len(a.index)),yticklabels=a.index,
           xlabel="Share favoring A",title="Fig 2: Unsupported commitment (%) — measured Qwen")
    for i in range(len(a.index)):
        for j in range(len(a.columns)):
            ax.text(j,i,f"{a.iloc[i,j]:.0f}%",ha="center",va="center")
    fig.colorbar(im,ax=ax);plt.show()
    percent_chart(s[s.test=="position"],"level",["correct","invalid"],"Fig 3: Evidence position")
    # Separate uncertainty from decisive evidence to catch excessive caution.
    prompt=s[s.test=="prompt"].copy()
    prompt["evidence_state"]=np.where(prompt.expected=="C","Incomplete","Decisive")
    display(prompt.groupby(["strategy","evidence_state"])[["correct","hold","invalid"]].mean().mul(100))
    prompt.groupby(["evidence_state","strategy"]).correct.mean().mul(100).unstack().plot.bar(ylim=(0,100),rot=0)
    plt.ylabel("Expected answer rate (%)");plt.title("Fig 4: Prompt comparison — measured Qwen");plt.show()
else:
    print("No Qwen results to graph. Run Section 4 with RUN_QWEN=True.")

### 7A. Read the conversation results

Figure 5 shows when a model first commits. Figure 6 follows individual answers as evidence arrives. A transition matrix counts changes from the last pre-proof answer to the final answer. We also calculate a correction rate only among conversations whose last pre-proof answer was the wrong cause. Invalid answers remain visible.

Example paths (not measured): C → C → C → B = waits; C → A → A → B = corrects; C → A → A → A = persists, if B is established.

In [ ]:
if not turns.empty:
    t=turns[turns["mode"]=="each-turn"].copy()
    paths=t.pivot(index="run",columns="turn",values="choice")
    meta=t.drop_duplicates("run").set_index("run")[["timing","strategy","ending","case"]]
    paths=paths.join(meta)
    paths["first_commit"]=paths.apply(lambda r:next((i for i in [1,2,3,4] if r[i] in ["A","B"]),5),axis=1)
    paths["hold"]=paths[[1,2,3]].eq("C").all(axis=1)
    paths["premature"]=paths[[1,2,3]].isin(["A","B"]).any(axis=1)
    paths["final_correct"]=paths[4].eq(paths.ending)
    paths["prior_wrong"]=paths[3].isin(["A","B"]) & paths[3].ne(paths.ending)
    paths["prior_invalid"]=paths[[1,2,3]].eq("INVALID").any(axis=1)
    counts=pd.crosstab(paths.timing,paths.first_commit).reindex(columns=[1,2,3,4,5],fill_value=0)
    counts.columns=["T1","T2","T3","T4","Never"]
    counts.div(counts.sum(axis=1),axis=0).mul(100).plot.bar(stacked=True,figsize=(8,4),rot=0)
    plt.ylabel("% of conversations");plt.title("Fig 5: First commitment turn — measured Qwen");plt.show()
    for timing in ["early","late","none"]:
        part=t[(t.timing==timing)&(t.strategy=="baseline")]
        mix=pd.crosstab(part.turn,part.choice,normalize="index").reindex(columns=["A","B","C","INVALID"],fill_value=0)*100
        mix.plot.bar(stacked=True,ylim=(0,100),figsize=(7,3),rot=0)
        plt.ylabel("% of responses");plt.title("Fig 6: "+timing+" lure, baseline — measured Qwen");plt.show()
    print("Transition counts by final cause (last pre-proof → after proof):")
    for end in ["A","B"]:
        sub=paths[paths.ending==end]
        print("Established cause:",end)
        display(pd.crosstab(sub[3],sub[4]))
    summary=paths.groupby(["timing","strategy"])[["hold","premature","final_correct","prior_invalid"]].mean()*100
    den=paths[paths.prior_wrong].groupby(["timing","strategy"]).size()
    num=paths[paths.prior_wrong & paths.final_correct].groupby(["timing","strategy"]).size()
    summary["correction_n"]=den
    summary["correction_%"]=num.reindex(den.index,fill_value=0)/den*100
    display(summary)
    print("NaN correction rate means no eligible wrong prior answers, not 0%.")
    final=turns[turns.turn==4].assign(correct=lambda d:d.choice.eq(d.expected))
    display(final.groupby(["mode","strategy"]).correct.agg(["mean","count"]))
    display(paths.head(10))
    paths.to_csv("foresight_v2_conversation_summary.csv")
else:
    print("No measured conversations yet. Run Section 5 with Qwen enabled.")

### 7B. Hypothesis coverage after human review

Fill the review sheet with 1 for yes, 0 for no, and leave uncertain entries blank. Use the review key to join the scores back to conditions. Never auto-fill missing reviews. Plot how often Qwen considered A, B, other plausible causes, or requested useful evidence. This is a separate metric from A/B/C choice accuracy.

In [ ]:
from pathlib import Path
REVIEWED_FILE="foresight_v2_human_review_completed.csv"
if not generation.empty and Path(REVIEWED_FILE).exists():
    reviewed=pd.read_csv(REVIEWED_FILE)
    key=pd.read_csv("foresight_v2_review_key.csv")
    graded=reviewed.merge(key,on="blind_id",validate="one_to_one").merge(
        generation[["review_id","share"]],on="review_id",validate="one_to_one")
    cols=["mentions_A","mentions_B","other_plausible","useful_evidence_request","unsupported_claim"]
    for col in cols:
        graded[col]=pd.to_numeric(graded[col],errors="raise")
        if not graded[col].dropna().isin([0,1]).all():
            raise ValueError("Review scores must be 0, 1, or blank")
    graded.groupby("share")[cols].mean().mul(100).plot.bar(ylim=(0,100),rot=0,figsize=(9,4))
    plt.ylabel("% of scored responses");plt.title("Fig 7: Human-reviewed hypothesis coverage");plt.show()
    display(graded.groupby("share")[cols].count())
else:
    print("Human review pending. No coverage scores invented.")

## 8. How to explain the results to class

**What if we get no effect?** That is a valid result. Maybe this Qwen model is resistant to the tested opinions. We need more scenarios before saying it is generally resistant.

**What if the model picks A more often as A becomes popular?** That suggests sensitivity to the supplied narrative under these controlled conditions. It does not prove how attention works internally.

**What if the model always says C?** Check the decisive evidence tests. If it still refuses to answer when proof arrives, the model is too cautious.

**What if better prompts help?** They are a possible practical safeguard. Test on held-out cases before making a recommendation.

**Why not just train Qwen?** We are studying behavior of a fixed model. Training is a different project.

**What still needs work?** More varied incidents; independent reviewers; matched token lengths; natural and duplicate comments; balanced cause labels; held-out scenarios; comparison across models; uncertainty intervals grouped by incident rather than treating repeated variants as independent. The pilot does not establish real-world reliability. The larger model comparison and downstream error propagation are planned extensions.

### One-minute explanation

We're looking at how popular opinions influence AI when the full story isn't known. Instead of teaching Qwen new facts, we give it controlled fictional incidents. We change the number of people supporting a guess, who supposedly said it, or when an important-looking clue arrives. We then measure whether Qwen jumps to a conclusion, overlooks other explanations or changes its answer when real proof is provided in the scenario. The point is to understand what causes the failure first, then see if better instructions can reduce it.

### Files produced
- foresight_v2_cases.csv — synthetic incident dataset
- foresight_v2_single.csv — real single-turn outputs (only if enabled)
- foresight_v2_turns.csv — real conversation history and responses
- foresight_v2_generation.csv — open-ended responses
- foresight_v2_human_review.csv and review_key.csv — for human scoring
- foresight_v2_conversation_summary.csv — measured decision paths

Keep the raw answers and model version with every report. Download the CSV files from Colab's Files panel.